In [ ]:
# =========================================================
# DAILY ERA5 HEAT INDEX EXTRACTION
# 11 AM – 5 PM | APRIL 2026
# DISTRICT-WISE DAILY DOWNLOAD
# =========================================================

# =========================================================
# 1. INSTALL PACKAGES
# =========================================================

# Run once in VSCode terminal:
# pip install earthengine-api geemap geopandas pandas numpy openpyxl


# =========================================================
# 2. IMPORTS
# =========================================================

import ee
import geemap
import geopandas as gpd
import pandas as pd
import numpy as np
from datetime import datetime, timedelta


# =========================================================
# 3. INITIALIZE EARTH ENGINE
# =========================================================

ee.Authenticate()
ee.Initialize(project='areca-farm')


# =========================================================
# 4. LOAD GEOJSON
# =========================================================

geojson_path = "../../assets/district.geojson"

gdf = gpd.read_file(geojson_path)

print("GeoJSON Loaded")


# =========================================================
# 5. REMOVE COMPLEX BOUNDARIES
# =========================================================
# Helps avoid incomplete edge pixels
# and speeds up computation
# =========================================================

gdf["geometry"] = gdf.geometry.simplify(0.01)

print("Geometry Simplified")


# =========================================================
# 6. CONVERT TO EARTH ENGINE
# =========================================================

districts = geemap.geopandas_to_ee(gdf)


# =========================================================
# 7. DATE RANGE
# =========================================================

start_date = datetime(2026, 4, 1)
end_date   = datetime(2026, 5, 1)


# =========================================================
# 8. ERA5 COLLECTION
# =========================================================

era5 = ee.ImageCollection("ECMWF/ERA5_LAND/HOURLY")


# =========================================================
# 9. HEAT INDEX FUNCTION
# =========================================================

def add_heat_index(img):

    # Temperature (°C)
    t = img.select('temperature_2m').subtract(273.15)

    # Dewpoint (°C)
    d = img.select('dewpoint_temperature_2m').subtract(273.15)

    # Relative Humidity
    rh = (
        d.expression(
            '''
            100 * (
                exp((17.625 * Td)/(243.04 + Td)) /
                exp((17.625 * T)/(243.04 + T))
            )
            ''',
            {
                'Td': d,
                'T': t
            }
        )
    )

    # Heat Index
    hi = (
        t.expression(
            '''
            -8.784695 +
            1.61139411*T +
            2.338549*RH -
            0.14611605*T*RH -
            0.012308094*(T**2) -
            0.016424828*(RH**2) +
            0.002211732*(T**2)*RH +
            0.00072546*T*(RH**2) -
            0.000003582*(T**2)*(RH**2)
            ''',
            {
                'T': t,
                'RH': rh
            }
        )
    ).rename('HI')

    return hi.copyProperties(img, ['system:time_start'])


# =========================================================
# 10. DAILY LOOP
# =========================================================

all_days = []

current = start_date

while current < end_date:

    next_day = current + timedelta(days=1)

    print(f"\nProcessing: {current.strftime('%Y-%m-%d')}")

    # -----------------------------------------------------
    # FILTER DAILY DATA
    # -----------------------------------------------------

    daily = (
        era5
        .filterDate(
            current.strftime('%Y-%m-%d'),
            next_day.strftime('%Y-%m-%d')
        )
        .filter(ee.Filter.calendarRange(11, 16, 'hour'))
        .map(add_heat_index)
    )

    # -----------------------------------------------------
    # DAILY 11AM–5PM MEAN
    # -----------------------------------------------------

    daily_hi = daily.mean()

    # -----------------------------------------------------
    # DISTRICT MEAN
    # -----------------------------------------------------

    stats = daily_hi.reduceRegions(
        collection=districts,
        reducer=ee.Reducer.mean(),
        scale=11132,
        tileScale=4
    )

    # -----------------------------------------------------
    # DOWNLOAD SMALL DAILY TABLE
    # -----------------------------------------------------

    features = stats.getInfo()['features']

    rows = []

    for f in features:

        props = f['properties']

        rows.append({
            'date': current.strftime('%Y-%m-%d'),
            'district': props.get('dtname'),
            'heat_index': props.get('mean')
        })

    df_day = pd.DataFrame(rows)

    print("Collected Successfully")

    all_days.append(df_day)

    current = next_day


# =========================================================
# 11. COMBINE ALL DAYS
# =========================================================

daily_df = pd.concat(all_days, ignore_index=True)

print("\nDaily Collection Complete")


# =========================================================
# 12. MONTHLY DISTRICT MEAN
# =========================================================

monthly_df = (
    daily_df
    .groupby('district', as_index=False)
    ['heat_index']
    .mean()
)

monthly_df.rename(
    columns={'heat_index': 'April_2026_HI'},
    inplace=True
)

print("\nMonthly Means Computed")


# =========================================================
# 13. MERGE BACK TO GEOJSON ATTRIBUTES
# =========================================================

final_df = gdf.merge(
    monthly_df,
    left_on='dtname',
    right_on='district',
    how='left'
)

# Remove geometry column
final_df = final_df.drop(columns='geometry')

print("\nFinal CSV Ready")


# =========================================================
# 14. EXPORT CSV
# =========================================================

output_csv = "../../district_heat_index_april_2026.csv"

final_df.to_csv(output_csv, index=False)

print(f"\nCSV Saved:\n{output_csv}")

GeoJSON Loaded
Geometry Simplified

Processing: 2026-04-01
Collected Successfully

Processing: 2026-04-02
Collected Successfully

Processing: 2026-04-03
Collected Successfully

Processing: 2026-04-04
Collected Successfully

Processing: 2026-04-05
Collected Successfully

Processing: 2026-04-06
Collected Successfully

Processing: 2026-04-07
Collected Successfully

Processing: 2026-04-08
Collected Successfully

Processing: 2026-04-09
Collected Successfully

Processing: 2026-04-10
Collected Successfully

Processing: 2026-04-11
Collected Successfully

Processing: 2026-04-12
Collected Successfully

Processing: 2026-04-13
Collected Successfully

Processing: 2026-04-14
Collected Successfully

Processing: 2026-04-15
Collected Successfully

Processing: 2026-04-16
Collected Successfully

Processing: 2026-04-17
Collected Successfully

Processing: 2026-04-18
Collected Successfully

Processing: 2026-04-19
Collected Successfully

Processing: 2026-04-20
Collected Successfully

Processing: 2026-04-21
C

In [4]:
print(len(all_days))

30


In [2]:
%pip install pathlib
from pathlib import Path
print(Path("../../assets/district.geojson").resolve())
print(Path("../../assets/district.geojson").exists())

Note: you may need to restart the kernel to use updated packages.
/home/root_1/Documents/CDL/IDS DRR/Heat-odisha/Data_extractor/assets/district.geojson
True
